# Tarea 3 – Manipulación de datos con DataFrames en PySpark

**Materia:** Datos Masivos  
**Alumno:** Sergio Cortes Cepeda

## Fuente de datos

Los datos utilizados provienen del repositorio Zenodo:
https://zenodo.org/records/7923702

Se trabajó con los meses de enero, febrero y marzo de los años 2019 a 2022.

## Introducción

En esta tarea se realiza el análisis de datos de vuelos de los años 2019 a 2022 utilizando DataFrames de PySpark.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count

spark = SparkSession.builder \
    .appName("Tarea3_Vuelos") \
    .getOrCreate()

## Explicacion de carga de datos

En esta parte tendre que cargar por archivo por año, debido a que el tamaño del peso de los archivos, para poder hacer manupulacion de datos.

In [3]:
df_2019_ene = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2019/flightlist_2019Enero.csv")

df_2019_feb = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2019/flightlist_2019Febrero.csv")

df_2019_mar = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2019/flightlist_2019Marzo.csv")

df_2019 = df_2019_ene.union(df_2019_feb).union(df_2019_mar)

In [4]:
df_2020_ene = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2020/flightlist_2020Enero.csv")

df_2020_feb = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2020/flightlist_2020Febrero.csv")

df_2020_mar = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2020/flightlist_2020Marzo.csv")

df_2020 = df_2020_ene.union(df_2020_feb).union(df_2020_mar)

In [5]:
df_2021_ene = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2021/flightlist_2021Enero.csv")

df_2021_feb = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2021/flightlist_2021Febrero.csv")

df_2021_mar = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2021/flightlist_2021Marzo.csv")

df_2021 = df_2021_ene.union(df_2021_feb).union(df_2021_mar)

In [6]:
df_2022_ene = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2022/flightlist_2022Enero.csv")

df_2022_feb = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2022/flightlist_2022Febrero.csv")

df_2022_mar = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2022/flightlist_2022Marzo.csv")

df_2022 = df_2022_ene.union(df_2022_feb).union(df_2022_mar)

In [7]:
df = df_2019.union(df_2020).union(df_2021).union(df_2022)

In [8]:
df.show(5)
df.printSchema()
print("Total de filas:", df.count())

+--------+------+------+------------+--------+------+-----------+--------------------+--------------------+--------------------+------------------+------------------+----------+------------------+------------------+----------+
|callsign|number|icao24|registration|typecode|origin|destination|           firstseen|            lastseen|                 day|        latitude_1|       longitude_1|altitude_1|        latitude_2|       longitude_2|altitude_2|
+--------+------+------+------------+--------+------+-----------+--------------------+--------------------+--------------------+------------------+------------------+----------+------------------+------------------+----------+
|   HVN19|  NULL|888152|        NULL|    NULL|  YMML|       LFPG|2018-12-31 00:43:...|2019-01-01 04:56:...|2019-01-01 00:00:...|-37.65948486328125|144.80442128282908|     304.8| 48.99531555175781| 2.610802283653846|    -53.34|
|  CCA839|  NULL|780ad1|        NULL|    NULL|  YMML|       LEBL|2018-12-31 00:53:...|2019-0

In [9]:
print("Filas 2019:", df_2019.count())
print("Filas 2020:", df_2020.count())
print("Filas 2021:", df_2021.count())
print("Filas 2022:", df_2022.count())

Filas 2019: 6434581
Filas 2020: 7535783
Filas 2021: 5480665
Filas 2022: 7503467


In [10]:
df = df.dropna(subset=["origin", "destination"])

In [11]:
print("Antes:", df.count())

df = df.dropna(subset=["origin", "destination"])

print("Después:", df.count())

Antes: 16078551
Después: 16078551


In [12]:
df.filter(col("origin") == "").count()
df.filter(col("destination") == "").count()

0

## Limpieza de datos

Se verificó la existencia de valores nulos y cadenas vacías en las columnas de origen y destino. No se encontraron registros con valores faltantes, por lo que no fue necesario aplicar procesos de limpieza adicionales. 

## Modificacion de columnas

In [13]:
df = df.withColumnRenamed("callsign", "vuelo") \
       .withColumnRenamed("origin", "aeropuerto_origen") \
       .withColumnRenamed("destination", "aeropuerto_destino")

df.select("vuelo", "aeropuerto_origen", "aeropuerto_destino", "day").show(5, truncate=False)

+------+-----------------+------------------+-------------------------+
|vuelo |aeropuerto_origen|aeropuerto_destino|day                      |
+------+-----------------+------------------+-------------------------+
|HVN19 |YMML             |LFPG              |2019-01-01 00:00:00+00:00|
|CCA839|YMML             |LEBL              |2019-01-01 00:00:00+00:00|
|CES219|YSSY             |EDDF              |2019-01-01 00:00:00+00:00|
|AEA040|LEMD             |LEMD              |2019-01-01 00:00:00+00:00|
|CXA825|YSSY             |LFPG              |2019-01-01 00:00:00+00:00|
+------+-----------------+------------------+-------------------------+
only showing top 5 rows


## Modificacion de datos

En esta sección se realizan transformaciones sobre el DataFrame, incluyendo renombrado de columnas, conversión de tipos de datos y creación de nuevas variables para facilitar el análisis.

In [14]:
from pyspark.sql.functions import col

df = df.withColumn("altitude_1", col("altitude_1").cast("double")) \
       .withColumn("altitude_2", col("altitude_2").cast("double"))

## Agregar nuevas columnas

Se generaron nuevas columnas calculadas a partir de la información original, como el año, mes, con el objetivo de enriquecer el análisis de los vuelos.

In [15]:
from pyspark.sql.functions import substring

df = df.withColumn("anio", substring(col("day"), 1, 4))

In [16]:
df = df.withColumn("mes", substring(col("day"), 6, 2))

## Análisis y agregaciones

Se realizaron agrupaciones por aeropuerto de origen y destino, así como por año, para identificar patrones en la actividad aérea.

In [18]:
##  Aeropuertos origen con mayor actividad 

from pyspark.sql.functions import col

df.groupBy("aeropuerto_origen") \
  .count() \
  .orderBy(col("count").desc()) \
  .show(10)

+-----------------+------+
|aeropuerto_origen| count|
+-----------------+------+
|             KORD|247645|
|             KLAX|204484|
|             KDFW|197076|
|             KATL|196795|
|             KDEN|182313|
|             KLAS|172965|
|             KPHX|162926|
|             KSEA|135888|
|             KEWR|132973|
|             KCLT|132797|
+-----------------+------+
only showing top 10 rows


In [19]:
#Vuelos por año

df.groupBy("anio") \
  .count() \
  .orderBy("anio") \
  .show()

+----+-------+
|anio|  count|
+----+-------+
|2019|3337203|
|2020|4507385|
|2021|3398320|
|2022|4835643|
+----+-------+



## Conclusión

El uso de DataFrames de PySpark permitió procesar un volumen considerable de datos de vuelos de manera eficiente. A través de la manipulación, transformación y análisis de los datos, se lograron identificar patrones importantes como los aeropuertos con mayor actividad y la distribución de vuelos por año. Esto demuestra la capacidad de PySpark para trabajar con datos masivos.